## Repeatability Requirements

To ensure this code is repeatable by other users, the following configurations and file structures are expected:

1.  **Google Drive Setup**: The notebook currently uses Google Drive paths. Users must mount their Google Drive in Colab and ensure video and annotation files are placed in a similar directory structure within their Drive, or modify the paths to local storage.
2.  **`MATCH_MAP` Configuration (Cell 4)**: The `MATCH_MAP` dictionary must be updated with the correct absolute paths to the downloaded `.mp4` video files. Ensure the sheet names match the tab names in your Excel files.
3.  **`EXCEL_PATH` Configuration (Cell 4)**: The `EXCEL_PATH` variable must point to the directory containing the annotation Excel files (e.g., `M01.xlsx`, `W01.xlsx`).
4.  **`OUTPUT_ROOT` Configuration (Cell 4)**: Configure this path to specify where the extracted video clips will be saved.
5.  **Excel Annotation File Structure**: The annotation Excel files are expected to have specific columns, including `Head event`, `Click`, and `Start`. Ensure your annotation files conform to this structure for correct parsing.

# ⚽ HAE Clip Extractor — Chapter 2 Dataset
Extracts 15-second clips from annotated WSL / Bundesliga / EPL / La Liga / Serie A / NCAA matches.

**Excel structure expected (one sheet per match, e.g. M01, W02):**

| N# | Head event | Start | Click | End | intentional... | ... |
|---|---|---|---|---|---|---|

- **`Head event`** — event label: `Header`, `Aerial Duel`, `PCE`, `Half 1 start`, `Match end`, etc.
- **`Click`** — the annotator's precise event timestamp used for clip centring (format `mm:ss`)
- `Start` / `End` — 5-second window around the click; used as fallback if `Click` is empty

Clips are saved as: `{match_id}_{gender}_{event}_{idx:04d}_t{seconds:.1f}s.mp4`

In [ ]:
# ── Cell 1: Mount Google Drive ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive mounted')

ValueError: mount failed

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
!pip install -q moviepy openpyxl pandas tqdm
print('✅ Dependencies ready')

✅ Dependencies ready


In [ ]:
# ── Cell 3: Imports ──────────────────────────────────────────────────────────
import os, re, warnings
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from moviepy.editor import VideoFileClip
warnings.filterwarnings('ignore')
print('✅ Imports OK')

/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':



✅ Imports OK


---
## ⚙️ Configuration — Edit This Cell Only

In [ ]:
# ── Cell 4: USER CONFIGURATION ───────────────────────────────────────────────
#
# MATCH_MAP maps: match_sheet_name → (mp4_drive_path, gender)
#
# Sheet names must exactly match the tab names in your Excel file
# (e.g. 'M01', 'W01' etc.)
#
# gender: 'Male' or 'Female' — used for output folder organisation
# ─────────────────────────────────────────────────────────────────────────────

MATCH_MAP = {
    # sheet_name : (full Drive path to .mp4,  gender)
    # 'M01': ('/content/drive/MyDrive/Chapter 2/Chapter2_Data/mp4 videos/MALE/M01.mp4',     'Male'), # Skipped as requested
    # 'M02': ('/content/drive/MyDrive/Chapter 2/Chapter2_Data/mp4 videos/MALE/m02.mp4',      'Male'), # Skipped as requested
    # 'M03': ('/content/drive/MyDrive/Chapter 2/Chapter2_Data/mp4 videos/MALE/M03.mp4',        'Male'), # Skipped as requested
    'M04': ('/content/drive/MyDrive/Chapter 2/Chapter2_Data/mp4 videos/MALE/M03.mp4',         'Male'),
    'M05': ('/content/drive/MyDrive/Chapter 2/Chapter2_Data/mp4 videos/MALE/M05.mp4',      'Male'),
    'M06': ('/content/drive/MyDrive/Chapter 2/Chapter2_Data/mp4 videos/MALE/m06.mp4',      'Male'),
    'M07': ('/content/drive/MyDrive/Chapter 2/Chapter2_Data/mp4 videos/MALE/m07.mp4',         'Male'),
    'M08': ('/content/drive/MyDrive/Chapter 2/Chapter2_Data/mp4 videos/MALE/M08.mp4',       'Male'),
    'M09': ('/content/drive/MyDrive/Chapter 2/Chapter2_Data/mp4 videos/MALE/M09.mp4',      'Male'),
    'M10': ('/content/drive/MyDrive/Chapter 2/Chapter2_Data/mp4 videos/MALE/m10.mp4',      'Male'),
    'W01': ('/content/drive/MyDrive/Chapter 2/Chapter2_Data/mp4 videos/FEMALE/W01.mp4',      'Female'),
    'W02': ('/content/drive/MyDrive/Chapter 2/Chapter2_Data/mp4 videos/FEMALE/W02.mp4',     'Female'),
    'W03': ('/content/drive/MyDrive/Chapter 2/Chapter2_Data/mp4 videos/FEMALE/W03.mp4',      'Female'),
    'W04': ('/content/drive/MyDrive/Chapter 2/Chapter2_Data/mp4 videos/FEMALE/W04.mp4',       'Female'),
    'W05': ('/content/drive/MyDrive/Chapter 2/Chapter2_Data/mp4 videos/FEMALE/W05.mp4',        'Female'),
    'W06': ('/content/drive/MyDrive/Chapter 2/Chapter2_Data/mp4 videos/FEMALE/W06.mp4',         'Female'),
    'W07': ('/content/drive/MyDrive/Chapter 2/Chapter2_Data/mp4 videos/FEMALE/w07.mp4',      'Female'),
    'W08': ('/content/drive/MyDrive/Chapter 2/Chapter2_Data/mp4 videos/FEMALE/W08.mp4',        'Female'),
    'W09': ('/content/drive/MyDrive/Chapter 2/Chapter2_Data/mp4 videos/FEMALE/W09.mp4',     'Female'),
    'W10': ('/content/drive/MyDrive/Chapter 2/Chapter2_Data/mp4 videos/FEMALE/w10.mp4',  'Female'),
}

# ── Path to your Chapter 2 annotations Excel file ────────────────────────────
EXCEL_PATH = '/content/drive/MyDrive/Chapter 2/Chapter2_Data/Annotations'

# ── Where to save the extracted clips on Drive ────────────────────────────────
OUTPUT_ROOT = '/content/drive/MyDrive/Chapter 2/Chapter2_Data/Extraction'

# ── Clip settings ─────────────────────────────────────────────────────────────
CLIP_DURATION  = 15.0    # total clip length in seconds
PRE_EVENT_SEC  = 7.5     # seconds BEFORE the 'Click' timestamp (event is centred by default)
                          # e.g. set to 5.0 for a 5s pre / 10s post window

# ── Event labels to extract ───────────────────────────────────────────────────
# Case-insensitive substring match against the 'Head event' column.
# Rows like 'Half 1 start', 'Half 1 end', 'Match end' are automatically excluded.
TARGET_EVENTS = [
    'header',
    'aerial duel',
    'pce',
    'ball 2 head',       # seen in W06
    'ball to head',
]

# ── Rows to always skip (non-events) ─────────────────────────────────────────
SKIP_LABELS = [
    'half 1 start', 'half 2 start', 'half 1 end', 'half 2 end',
    'match start', 'match end', 'environmental',
]

print('✅ Configuration loaded')
print(f'   Clip : {CLIP_DURATION}s  ({PRE_EVENT_SEC}s pre / {CLIP_DURATION - PRE_EVENT_SEC}s post event)')
print(f'   Target events : {TARGET_EVENTS}')
print(f'   Matches mapped: {len(MATCH_MAP)}')


✅ Configuration loaded
   Clip : 15.0s  (7.5s pre / 7.5s post event)
   Target events : ['header', 'aerial duel', 'pce', 'ball 2 head', 'ball to head']
   Matches mapped: 17


---
## 🔧 Core Functions

In [ ]:
# ── Cell 5: Helpers ───────────────────────────────────────────────────────────

def parse_timestamp(ts) -> float:
    """
    Convert mm:ss, mm:ss.ms, HH:MM:SS, or plain seconds to total float seconds.
    Returns None if unparseable.
    """
    if ts is None:
        return None
    if isinstance(ts, (int, float)):
        return float(ts) if not pd.isna(ts) else None
    ts = str(ts).strip()
    if not ts:
        return None
    parts = ts.split(':')
    try:
        if len(parts) == 2:
            return int(parts[0]) * 60 + float(parts[1])
        elif len(parts) == 3:
            return int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2])
        else:
            return float(ts)
    except (ValueError, TypeError):
        return None


def is_target_event(label, targets, skips) -> bool:
    """Return True if label matches a target and is not a skip label."""
    if not isinstance(label, str):
        return False
    ll = label.strip().lower()
    if any(s in ll for s in skips):
        return False
    return any(t in ll for t in targets)


def safe_fn(text) -> str:
    """Strip filesystem-illegal characters for use in filenames."""
    return re.sub(r'[\\/:*?"<>|\s]+', '_', str(text)).strip('_')


print('✅ Helpers defined')

✅ Helpers defined


In [ ]:
# ── Cell 6: Annotation loader ─────────────────────────────────────────────────

def load_sheet(xlsx_path: str, sheet: str, gender: str) -> pd.DataFrame:
    """
    Load a single match sheet and return a filtered DataFrame ready for extraction.

    Column mapping (your file):
      'Head event'  → event label
      'Click'       → precise annotator timestamp (primary)
      'Start'       → fallback if Click is empty

    Returns DataFrame with added columns:
      _seconds      → float seconds of the event
      _event_clean  → cleaned event label string
      _gender       → gender string
      _match        → sheet/match ID
    """
    # Load the first sheet by default, assuming each Excel file is a single match's annotations
    df = pd.read_excel(xlsx_path, header=0)

    # Normalise column names
    df.columns = [str(c).strip() for c in df.columns]

    # Identify the key columns (case-insensitive search)
    col_map = {c.lower(): c for c in df.columns}

    # Define potential event column names to check
    POTENTIAL_EVENT_COL_NAMES = ['head event', 'event', 'category'] # Add more if necessary

    event_col = None
    for potential_col in POTENTIAL_EVENT_COL_NAMES:
        if col_map.get(potential_col):
            event_col = col_map.get(potential_col)
            break

    click_col  = col_map.get('click')
    start_col  = col_map.get('start')

    if not event_col:
        raise ValueError(f'Sheet "{sheet}": no event column found from {POTENTIAL_EVENT_COL_NAMES}. Columns: {list(df.columns)}')
    if not click_col and not start_col:
        raise ValueError(f'Sheet "{sheet}": no "Click" or "Start" column found.')

    # Drop fully empty rows
    df = df.dropna(how='all').reset_index(drop=True)

    # Filter to target events
    df['_event_clean'] = df[event_col].astype(str).str.strip()
    mask = df['_event_clean'].apply(
        lambda x: is_target_event(x, TARGET_EVENTS, SKIP_LABELS)
    )
    df = df[mask].copy()

    if df.empty:
        return df

    # Resolve timestamp: prefer 'Click', fall back to 'Start'
    def resolve_ts(row):
        ts = None
        if click_col:
            ts = parse_timestamp(row.get(click_col))
        if ts is None and start_col:
            ts = parse_timestamp(row.get(start_col))
            if ts is not None:
                ts += 5.0   # Click ≈ Start + 5s based on your annotation convention
        return ts

    df['_seconds'] = df.apply(resolve_ts, axis=1)
    df = df.dropna(subset=['_seconds']).reset_index(drop=True)

    df['_gender'] = gender
    df['_match']  = sheet

    print(f'   📄 {sheet} [{gender}]: {len(df)} target events loaded')
    return df


print('✅ Sheet loader defined')

✅ Sheet loader defined


In [ ]:
# ── Cell 7: Clip extractor ────────────────────────────────────────────────────

def extract_clips(video_path: str, annotations: pd.DataFrame,
                  output_dir: str) -> dict:
    """
    Extract 15-second clips centred on the Click timestamp.

    Output filename:
      {match}_{gender}_{event}_{idx:04d}_t{seconds:.1f}s.mp4

    Returns summary dict {extracted, skipped, errors}.
    """
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    summary = {'extracted': 0, 'skipped': 0, 'errors': 0}

    print(f'\n🎬 {Path(video_path).name}  →  {output_dir}')

    with VideoFileClip(str(video_path)) as video:
        total_dur = video.duration
        print(f'   Duration: {total_dur:.1f}s  |  FPS: {video.fps}')

        for idx, row in tqdm(annotations.iterrows(),
                              total=len(annotations),
                              desc='  Clips', unit='clip'):

            event_sec   = row['_seconds']
            event_label = safe_fn(row['_event_clean'])
            match_id    = row['_match']
            gender      = row['_gender']

            # ── Clip window ──────────────────────────────────────────────────
            start_sec = max(0.0, event_sec - PRE_EVENT_SEC)
            end_sec   = start_sec + CLIP_DURATION

            # Clamp to video end
            if end_sec > total_dur:
                end_sec   = total_dur
                start_sec = max(0.0, end_sec - CLIP_DURATION)

            if (end_sec - start_sec) < 5.0:
                print(f'  ⚠️  Skipping t={event_sec:.1f}s — too close to video boundary')
                summary['skipped'] += 1
                continue

            # ── Output path ──────────────────────────────────────────────────
            fname = f'{match_id}_{gender}_{event_label}_{idx:04d}_t{event_sec:.1f}s.mp4'
            out_path = Path(output_dir) / fname

            if out_path.exists():
                summary['skipped'] += 1
                continue

            # ── Extract ──────────────────────────────────────────────────────
            try:
                clip = video.subclip(start_sec, end_sec)
                clip.write_videofile(
                    str(out_path),
                    codec='libx264',
                    audio=False,
                    fps=video.fps,
                    preset='fast',
                    ffmpeg_params=['-crf', '23'],
                    logger=None
                )
                summary['extracted'] += 1
            except Exception as e:
                print(f'  ❌ Error at t={event_sec:.1f}s: {e}')
                summary['errors'] += 1

    return summary


print('✅ Clip extractor defined')

✅ Clip extractor defined


---
## 🔍 Step 1 — Preview Annotations (Run Before Extracting)

In [ ]:
# ── Cell 8: Preview annotations across all sheets ─────────────────────────────
# Loads the Excel, counts events per match, shows a summary — NO video needed.
# Run this first to verify everything looks correct.

print(f'📊 Loading annotations from directory: {EXCEL_PATH}\n')

all_annotations = {}  # match_id → DataFrame
preview_rows = []

for match_id, (mp4_path, gender) in MATCH_MAP.items():
    excel_file_path = os.path.join(EXCEL_PATH, f'{match_id}.xlsx') # Construct full path

    if not os.path.exists(excel_file_path):
        print(f'  ⚠️  Excel file "{excel_file_path}" not found — skipping match {match_id}')
        continue
    if not os.path.isfile(excel_file_path):
        print(f'  ⚠️  Path "{excel_file_path}" is not a file — skipping match {match_id}')
        continue

    try:
        # Call load_sheet with the specific excel_file_path and match_id as the sheet name
        # Assuming each Excel file MXX.xlsx contains a sheet also named MXX
        df = load_sheet(excel_file_path, match_id, gender)
        all_annotations[match_id] = df

        if not df.empty:
            counts = df['_event_clean'].value_counts()
            for evt, cnt in counts.items():
                preview_rows.append({'Match': match_id, 'Gender': gender,
                                      'Event': evt, 'Count': cnt})
    except Exception as e:
        print(f'  ❌ {match_id}: {e}')

if preview_rows:
    summary_df = pd.DataFrame(preview_rows)
    pivot = summary_df.pivot_table(index=['Match','Gender'], columns='Event',
                                    values='Count', fill_value=0, aggfunc='sum')
    pivot['TOTAL'] = pivot.sum(axis=1)
    print('\n📋 Event counts per match:')
    print(pivot.to_string())
    print(f'\n✅ Total clips to extract: {pivot["TOTAL"].sum()}')
else:
    print('No target events found — check TARGET_EVENTS or Excel file names/structure.')

📊 Loading annotations from directory: /content/drive/MyDrive/Chapter 2/Chapter2_Data/Annotations

   📄 M04 [Male]: 92 target events loaded
   📄 M05 [Male]: 64 target events loaded
   📄 M06 [Male]: 57 target events loaded
   📄 M07 [Male]: 60 target events loaded
   📄 M08 [Male]: 129 target events loaded
   📄 M09 [Male]: 95 target events loaded
   📄 M10 [Male]: 116 target events loaded
   📄 W01 [Female]: 77 target events loaded
   📄 W02 [Female]: 76 target events loaded
   📄 W03 [Female]: 90 target events loaded
   📄 W04 [Female]: 62 target events loaded
   📄 W05 [Female]: 65 target events loaded
   📄 W06 [Female]: 91 target events loaded
   📄 W07 [Female]: 69 target events loaded
   📄 W08 [Female]: 67 target events loaded
  ❌ W09: Sheet "W09": no event column found from ['head event', 'event', 'category']. Columns: ['1', 'Half 1 start', '08:59', '09:02', '09:05', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10']
   📄 W10 [Female]: 61 target events load

---
## 🚀 Step 2 — Extract All Clips

In [ ]:
# ── Cell 9: MAIN EXTRACTION PIPELINE ─────────────────────────────────────────
# Loops all matches, extracts clips, organises into:
#   OUTPUT_ROOT / {Gender} / {sheet} / *.mp4

all_summaries = []
total_extracted = total_skipped = total_errors = 0

for sheet, (mp4_path, gender) in MATCH_MAP.items():
    print('=' * 65)

    if sheet not in all_annotations:
        print(f'⏭  {sheet}: no annotations loaded — skipping')
        continue

    annotations = all_annotations[sheet]
    if annotations.empty:
        print(f'⏭  {sheet}: 0 target events — skipping')
        continue

    mp4 = Path(mp4_path)
    if not mp4.exists():
        print(f'❌  {sheet}: Video not found → {mp4_path}')
        continue

    output_dir = os.path.join(OUTPUT_ROOT, gender, sheet)

    try:
        summary = extract_clips(str(mp4), annotations, output_dir)
        summary.update({'match': sheet, 'gender': gender,
                         'events': len(annotations)})
        all_summaries.append(summary)
        total_extracted += summary['extracted']
        total_skipped   += summary['skipped']
        total_errors    += summary['errors']
        print(f'  ✅ {sheet}: {summary["extracted"]} extracted, '
              f'{summary["skipped"]} skipped, {summary["errors"]} errors')
    except Exception as e:
        print(f'  ❌ {sheet} failed: {e}')

print('\n' + '=' * 65)
print('🏁 PIPELINE COMPLETE')
print(f'   Extracted : {total_extracted}')
print(f'   Skipped   : {total_skipped}')
print(f'   Errors    : {total_errors}')
print(f'   Output    : {OUTPUT_ROOT}')


🎬 M03.mp4  →  /content/drive/MyDrive/Chapter 2/Chapter2_Data/Extraction/Male/M04
   Duration: 6037.7s  |  FPS: 30.0


  Clips: 100%|██████████| 92/92 [00:00<00:00, 1764.86clip/s]

  ✅ M04: 0 extracted, 92 skipped, 0 errors

🎬 M05.mp4  →  /content/drive/MyDrive/Chapter 2/Chapter2_Data/Extraction/Male/M05


   Duration: 5926.2s  |  FPS: 30.0


  Clips: 100%|██████████| 64/64 [00:00<00:00, 1010.74clip/s]


...

  ✅ W07: 0 extracted, 69 skipped, 0 errors

🎬 W08.mp4  →  /content/drive/MyDrive/Chapter 2/Chapter2_Data/Extraction/Female/W08
   Duration: 6641.9s  |  FPS: 30.0


  Clips: 100%|██████████| 67/67 [31:15<00:00, 28.00s/clip]


  ✅ W08: 52 extracted, 15 skipped, 0 errors
⏭  W09: no annotations loaded — skipping

🎬 w10.mp4  →  /content/drive/MyDrive/Chapter 2/Chapter2_Data/Extraction/Female/W10
   Duration: 7026.8s  |  FPS: 30.0


  Clips: 100%|██████████| 61/61 [33:23<00:00, 32.84s/clip]

  ✅ W10: 61 extracted, 0 skipped, 0 errors

🏁 PIPELINE COMPLETE
   Extracted : 763
   Skipped   : 508
   Errors    : 0
   Output    : /content/drive/MyDrive/Chapter 2/Chapter2_Data/Extraction


In [ ]:
# ── Cell 10: Per-match summary + CSV export ───────────────────────────────────
if all_summaries:
    s_df = pd.DataFrame(all_summaries)[['match','gender','events','extracted','skipped','errors']]
    print(s_df.to_string(index=False))
    os.makedirs(OUTPUT_ROOT, exist_ok=True)
    csv_out = os.path.join(OUTPUT_ROOT, 'extraction_summary.csv')
    s_df.to_csv(csv_out, index=False)
    print(f'\n💾 Summary → {csv_out}')

match gender  events  extracted  skipped  errors
  M04   Male      92          0       92       0
  M05   Male      64          0       64       0
  M06   Male      57          0       57       0
  M07   Male      60          0       60       0
  M08   Male     129         69       60       0
  M09   Male      95         95        0       0
  M10   Male     116        116        0       0
  W01 Female      77         77        0       0
  W02 Female      76         76        0       0
  W03 Female      90         90        0       0
  W04 Female      62         62        0       0
  W05 Female      65         65        0       0
  W06 Female      91          0       91       0
  W07 Female      69          0       69       0
  W08 Female      67         52       15       0
  W10 Female      61         61        0       0

💾 Summary → /content/drive/MyDrive/Chapter 2/Chapter2_Data/Extraction/extraction_summary.csv


In [ ]:
# ── Cell 11: Build full clip manifest ────────────────────────────────────────
# Walks OUTPUT_ROOT, records every .mp4 with metadata parsed from filename.
# Output: clip_manifest.csv — ready to feed as a dataloader annotation file.

records = []
for root, dirs, files in os.walk(OUTPUT_ROOT):
    for fname in sorted(files):
        if not fname.endswith('.mp4'):
            continue
        fpath = os.path.join(root, fname)
        # Parse from filename: {match}_{gender}_{event}_{idx}_t{sec}s.mp4
        stem = fname.replace('.mp4', '')
        parts = stem.split('_')
        match_id = parts[0] if len(parts) > 0 else ''
        gender   = parts[1] if len(parts) > 1 else ''
        records.append({
            'clip_path': fpath,
            'filename' : fname,
            'match_id' : match_id,
            'gender'   : gender,
            'size_mb'  : round(os.path.getsize(fpath) / 1e6, 2),
        })

if records:
    manifest = pd.DataFrame(records)
    manifest_path = os.path.join(OUTPUT_ROOT, 'clip_manifest.csv')
    manifest.to_csv(manifest_path, index=False)
    print(f'📋 Manifest saved → {manifest_path}')
    print(f'   Total clips : {len(manifest)}')
    print(f'   Total size  : {manifest["size_mb"].sum():.1f} MB')
    print('\n   By gender:')
    print(manifest['gender'].value_counts().rename_axis('Gender').reset_index(name='Clips').to_string(index=False))
else:
    print('No clips found in output root yet.')

📋 Manifest saved → /content/drive/MyDrive/Chapter 2/Chapter2_Data/Extraction/clip_manifest.csv
   Total clips : 2528
   Total size  : 10376.2 MB

   By gender:
Gender  Clips
  Male    926
Female    662
    no    470
Header    366
Aerial    104


In [ ]:
# ── Cell 12: (Optional) Extract a SINGLE match for testing ───────────────────
# Useful for checking one match before running the full pipeline.
# Change SHEET and VIDEO_PATH, then run this cell alone.

TEST_SHEET  = 'M01'
TEST_VIDEO  = '/content/drive/MyDrive/HAE_Data/Videos/M01_Dortmund_Schalke_2015.mp4'
TEST_GENDER = 'Male'

df_test = load_sheet(EXCEL_PATH, TEST_SHEET, TEST_GENDER)
print(df_test[['_event_clean', '_seconds']].head(10).to_string(index=False))

# Uncomment to actually extract:
# out = extract_clips(TEST_VIDEO, df_test,
#                     os.path.join(OUTPUT_ROOT, TEST_GENDER, TEST_SHEET + '_test'))
# print(out)